In [1]:
!pip install spacy scikit-learn
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 26.1 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import spacy
from itertools import combinations
from collections import Counter
from sklearn.svm import SVC

In [3]:
nlp = spacy.load("en_core_web_sm")

In [5]:
def get_dependency_path(ent1, ent2):
    token1 = ent1.root
    token2 = ent2.root
    # Find ancestors of both entity tokens
    ancestors1 = [token1] + list(token1.ancestors)
    ancestors2 = [token2] + list(token2.ancestors)
    # Find the lowest common ancestor
    common = next(
        (t for t in ancestors1 if t in ancestors2),
        None
    )
    if common is None:
        return []
    # Path from first entity to common ancestor
    path1 = []
    token = token1
    while token != common:
        path1.append(token.lemma_.lower())
        path1.append("<" + token.dep_ + ">")
        token = token.head
    path1.append(common.lemma_.lower())
    # Path from common ancestor to second entity
    path2 = []
    token = token2
    while token != common:
        path2.append(token.lemma_.lower())
        path2.append(">" + token.dep_ + ">")
        token = token.head
    return path1 + list(reversed(path2))

In [7]:
def get_subsequences(path, max_length=3):
    features = []
    for length in range(1, min(max_length, len(path)) + 1):
        for subsequence in combinations(path, length):
            features.append(subsequence)
    return Counter(features)

In [9]:
def subsequence_kernel(path1, path2):
    features1 = get_subsequences(path1)
    features2 = get_subsequences(path2)
    common_features = set(features1) & set(features2)
    return sum(
        features1[f] * features2[f]
        for f in common_features
    )

In [11]:
training_data = [
    ("Steve Jobs founded Apple.", "FOUNDED"),
    ("Larry Page founded Google.", "FOUNDED"),
    ("Satya Nadella works at Microsoft.", "WORKS_FOR"),
    ("Sundar Pichai works at Google.", "WORKS_FOR"),
    ("Paris is located in France.", "LOCATED_IN"),
    ("Mumbai is located in India.", "LOCATED_IN")
]
train_paths = []
train_labels = []
for sentence, relation in training_data:
    doc = nlp(sentence)
    entities = list(doc.ents)
    if len(entities) >= 2:
        path = get_dependency_path(
            entities[0], entities[1]
        )
        if path:
            train_paths.append(path)
            train_labels.append(relation)

In [13]:
n = len(train_paths)
kernel_matrix = [
[
subsequence_kernel(train_paths[i], train_paths[j])
for j in range(n)
]
for i in range(n)
]
# Train the relation classifier
model = SVC(kernel="precomputed")
model.fit(kernel_matrix, train_labels)

SVC(kernel='precomputed')

In [15]:
def extract_relations(text):
    doc = nlp(text)
    entities = list(doc.ents)
    results = []
    for ent1, ent2 in combinations(entities, 2):
        path = get_dependency_path(ent1, ent2)
        if not path:
            continue
        test_kernel = [
            subsequence_kernel(path, train_path)
            for train_path in train_paths
        ]
        relation = model.predict([test_kernel])[0]
        results.append({
            "Entity 1": ent1.text,
            "Entity 2": ent2.text,
            "Dependency Path": " ".join(path),
            "Relation": relation
        })
    return results

In [17]:
text = (
"Steve Jobs founded Apple. "
"Satya Nadella works at Microsoft. "
"Paris is located in France."
)
for result in extract_relations(text):
    print(result)

{'Entity 1': 'Steve Jobs', 'Entity 2': 'Apple', 'Dependency Path': 'jobs <nsubj> found >dobj> apple', 'Relation': np.str_('FOUNDED')}
{'Entity 1': 'Satya Nadella', 'Entity 2': 'Microsoft', 'Dependency Path': 'nadella <nsubj> work >prep> at >pobj> microsoft', 'Relation': np.str_('WORKS_FOR')}
{'Entity 1': 'Paris', 'Entity 2': 'France', 'Dependency Path': 'paris <nsubjpass> locate >prep> in >pobj> france', 'Relation': np.str_('LOCATED_IN')}
